In [0]:
# Environment selection as dropdown
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="fq_dev",
    choices=["fq_dev", "fq_test", "fq_prod"],
    label="Select environment"
)

# Source selection as combobox
dbutils.widgets.combobox(
    name="source",
    defaultValue="POSIST",
    choices=["POSIST", "NETSUITE", "other"],
    label="Source"
)

# Domain selection as combobox
dbutils.widgets.combobox(
    name="domain",
    defaultValue="sales",
    choices=["discount", "sales", "cost"],
    label="Domain"
)

environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]

print(f"Environment: {environment}")
print(f"Source: {source}")
print(f"Domain: {domain}")

In [0]:
%sql select sum(net_amount_before_discount), sum(bill_discount), sum(total_items_discount), sum(tax_amount), sum(gross_amount) from fq_dev_catalog.silver.sales_bills where deployment_name = 'ALBAIK - SQ DB05 - AL MAJAZ - 1005005' and business_date = '2026-01-10' and is_void = false

In [0]:
%sql select * from fq_dev_catalog.silver.sales_bills where deployment_name = 'ALBAIK - SQ DB05 - AL MAJAZ - 1005005' and business_date = '2026-01-10' and is_void = false limit 1

In [0]:
%sql
-- ALTER TABLE fq_dev_catalog.gold.sales_channel_wise RENAME COLUMN net_sales TO net_amount_before_discount;

In [0]:
%sql
-- alter table fq_dev_catalog.gold.sales_channel_wise add columns (created_ts TIMESTAMP, last_modified_ts TIMESTAMP);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fq_dev_catalog.gold.sales_channel_wise (
    business_date DATE,
    deployment_name STRING,
    order_type STRING,
    tab_name STRING,
    total_bills INT,
    net_amount_before_discount DECIMAL(18,3),
    bill_discount DECIMAL(18, 3),
    items_discount DECIMAL(18, 3),
    total_discount DECIMAL(18,3),
    net_amount_after_discount DECIMAL(18,3),
    tax_amount DECIMAL(18,3),
    gross_sales DECIMAL(18,3),
    created_ts TIMESTAMP,
    last_modified_ts TIMESTAMP
)
PARTITIONED BY (
    deployment_name
)
TBLPROPERTIES (
    delta.columnMapping.mode = 'name',
    delta.enableChangeDataFeed = true,
    delta.autoOptimize.optimizeWrite = true,
    delta.autoOptimize.autoCompact = true
);

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col, countDistinct, sum, current_timestamp
import time

def upsert_channel_sales(microBatchDF, batchId):
    # Deduplicate: keep only the latest change per key
    windowSpec = Window.partitionBy(
        "ng_id"
    ).orderBy(col("_commit_version").desc())
    deduped = (
        microBatchDF.withColumn("rank", dense_rank().over(windowSpec))
        .filter("rank = 1 and _change_type != 'update_preimage'")
        .drop("rank", "_commit_version")
    )

    # Handle deletes: filter out deleted records for aggregation
    active = deduped.filter(col("_change_type") != "delete").filter(col("is_void") == False)

    # Aggregate to gold schema
    gold_df = (
        active.groupBy(
            "order_type", "tab_name", "deployment_name", "business_date"
        )
        .agg(
            countDistinct("bill_id").alias("total_bills"),
            sum("net_amount_before_discount").alias("net_amount_before_discount"),
            sum("tax_amount").alias("tax_amount"),
            sum("gross_amount").alias("gross_sales"),
            sum('bill_discount').alias('bill_discount'),
            sum('total_items_discount').alias('items_discount'),
            sum("net_amount_after_discount").alias("net_amount_after_discount")
        )
        .withColumn("total_discount", col("bill_discount") + col("items_discount"))
        .withColumn("created_ts", current_timestamp())
        .withColumn("last_modified_ts", current_timestamp())
    )

    # Upsert to gold table
    (DeltaTable.forName(spark, "fq_dev_catalog.gold.sales_channel_wise").alias("target")
    .merge(
        gold_df.alias("source"),
        "source.business_date = target.business_date AND "
        "source.deployment_name = target.deployment_name AND "
        "source.order_type = target.order_type AND "
        "source.tab_name = target.tab_name"
    )
    .whenMatchedUpdate(
        set={
            "total_bills": "source.total_bills",
            "net_amount_before_discount": "source.net_amount_before_discount",
            "tax_amount": "source.tax_amount",
            "gross_sales": "source.gross_sales",
            "bill_discount": "source.bill_discount",
            "items_discount": "source.items_discount",
            "net_amount_after_discount": "source.net_amount_after_discount",
            "total_discount": "source.total_discount",
            "last_modified_ts": "source.last_modified_ts"
        }
    )
    .whenNotMatchedInsert(
        values={
            "business_date": "source.business_date",
            "deployment_name": "source.deployment_name",
            "order_type": "source.order_type",
            "tab_name": "source.tab_name",
            "total_bills": "source.total_bills",
            "net_amount_before_discount": "source.net_amount_before_discount",
            "tax_amount": "source.tax_amount",
            "gross_sales": "source.gross_sales",
            "bill_discount": "source.bill_discount",
            "items_discount": "source.items_discount",
            "net_amount_after_discount": "source.net_amount_after_discount",
            "total_discount": "source.total_discount",
            "created_ts": "source.created_ts",
            "last_modified_ts": "source.last_modified_ts"
        }
    )
    .execute())

(spark.readStream
    .option("readChangeData", "true")
    .option("startingVersion", 35)
    .option("schemaTrackingLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_sales_channel_wise/sales_bills_schema_tracking') # schema-evolution
    .table("fq_dev_catalog.silver.sales_bills")
    .writeStream
    .foreachBatch(upsert_channel_sales)
    .option('mergeSchema', True) # schema-evolution
    .option("checkpointLocation", f'{checkpoint}/{source}/{domain}/streaming/checkpoint_sales_channel_wise')
    .trigger(availableNow=True)
    .start()
).awaitTermination()

time.sleep(20)

In [0]:
%sql
SELECT 
  count(distinct order_type) as order_type_cardinality,
  count(distinct tab_name) as tab_name_cardinality,
  count(distinct deployment_name) as deployment_name_cardinality,
  count(distinct business_date) as business_date_cardinality,
  COUNT(*) AS total_rows
FROM fq_dev_catalog.gold.sales_channel_wise;

In [0]:
%sql
OPTIMIZE fq_dev_catalog.gold.sales_channel_wise ZORDER BY (business_date, tab_name); -- high cardinality non-partitioned column first

In [0]:
%sql
select * from fq_dev_catalog.gold.sales_channel_wise where bill_discount > 0 and items_discount > 0 limit 1

In [0]:
for query in spark.streams.active:
    query.stop()